# OptimusPrimus: training, inference and detection model

Run from the project root (the folder containing `Data/`). The training cell is skipped unless `training_run = True`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from OptimusPrimus import (OptimusPrimus, OptimusPrimusTraining, load_detection_logs, calibrate_detection_model,
                           binned_recall_precision, fold_detection, merge_bins_auto, aggregate_bins, detection_chi2)
from mineral_utils import slice_spectrum

## Configuration

In [ ]:
IMAGE_SPEC = "ARCI URAS26F2"          # folder Data/images/<IMAGE_SPEC>/ (tiles in its 'tiles' subfolder)

# Overrides of the defaults in OptimusPrimus.py (only the keys given here change)
IMAGE_CONFIG = {}                       # e.g. {'pixel_resolution_um_per_px': 0.345}
SEG_MODEL_CONFIG = {}                   # e.g. {'threshold': 0.1}  (operating point for inference and evaluation)
CLASS_MODEL_CONFIG = {}                 # e.g. {'threshold': 0.55}

# Checkpoints. None = the models trained on IMAGE_SPEC (seg_model_<encoder>_<IMAGE_SPEC>, class_model_<encoder>_<IMAGE_SPEC>).
# To apply models trained on another image set, give their names, e.g. "seg_model_efficientnet-b7_ARCI URAS26F2".
SEG_MODEL_SPEC = None
CLASS_MODEL_SPEC = None

## Training and evaluation (optional)

In [ ]:
training_run = False

if training_run:
    trainer = OptimusPrimusTraining(f"Data/images/{IMAGE_SPEC}", IMAGE_SPEC,
                                    training_parameters={},          # e.g. {'seg_epochs': 200}
                                    image_config=IMAGE_CONFIG, seg_model_config=SEG_MODEL_CONFIG,
                                    class_model_config=CLASS_MODEL_CONFIG, parallel=True)
    trainer.perform_seg_training()
    trainer.perform_class_training()     # candidates from the trained segmentation model; negatives='rims' (default), 'no_rims' or 'unmatched': see the comparison section below

    # binned recall/precision (csv) and the detection-model logs (json), next to the checkpoints
    eff_seg = trainer.evaluate_binned_efficiency('seg')
    eff_seg_class = trainer.evaluate_binned_efficiency('seg_class')
    display(eff_seg_class['efficiency_table'])

## Inference

In [ ]:
track_detector = OptimusPrimus(IMAGE_SPEC, SEG_MODEL_SPEC, CLASS_MODEL_SPEC, image_config=IMAGE_CONFIG,
                               seg_model_config=SEG_MODEL_CONFIG, class_model_config=CLASS_MODEL_CONFIG, parallel=True)

print(f"Tiles folder         : {track_detector.image_path}")
print(f"z-stacks / area      : {len(track_detector.image_groups)} / {track_detector.tot_area_cm2:.4f} cm^2")
print(f"Segmentation model   : {track_detector.seg_model_path}")
print(f"Classification model : {track_detector.cls_model_path}")
print(f"Device               : {track_detector.device}")

In [ ]:
track_detector.perform_full_inference()          # thresholds from the model configurations
track_detector.seg_cls_ellipses.head()

In [ ]:
seg_stats = track_detector.get_track_distributions('seg')
seg_cls_stats = track_detector.get_track_distributions('seg_class')
pd.DataFrame({'seg': seg_stats, 'seg_class': seg_cls_stats})

Saved results can be reloaded without running the networks again:

In [ ]:
track_detector.inference_from_file('seg')
track_detector.inference_from_file('seg_class')
track_detector.get_track_distributions('seg_class')

## Classifier negatives: comparison (optional)

How the classifier's training samples are labelled (`negatives` in `perform_class_training`):
- `'rims'` (default): positives are the annotated tracks, negatives are all predicted pixels outside the annotations (rims around real tracks and false-positive components);
- `'no_rims'`: as `'rims'`, but only predicted components that do not touch an annotation are negatives;
- `'unmatched'`: every predicted component is labelled with the IoU matching used in the evaluation (positive if matched to an annotated track), so the patches are those seen at inference.

Each variant is trained under its own name (separate checkpoints, shared train/val/test split) and compared at equal precision with `classifier_operating_curve`. The printed summary shows how many negatives touch an annotation.

In [ ]:
COMPARE_NEGATIVES = False

if COMPARE_NEGATIVES:
    SEG_CKPT = track_detector.seg_model_path                     # trained segmentation model shared by the variants
    curves = {}
    for variant in ('rims', 'no_rims', 'unmatched'):
        tr = OptimusPrimusTraining(f"Data/images/{IMAGE_SPEC}", f"{IMAGE_SPEC}_{variant}", image_config=IMAGE_CONFIG,
                                   seg_model_config=SEG_MODEL_CONFIG, class_model_config=CLASS_MODEL_CONFIG, parallel=True)
        tr.perform_class_training(seg_model_path=SEG_CKPT, negatives=variant)
        curves[variant] = tr.classifier_operating_curve(seg_model_path=SEG_CKPT)

In [ ]:
if COMPARE_NEGATIVES:
    TARGET_PRECISION = 0.9
    fig, ax = plt.subplots(figsize=(5, 4))
    for variant, df in curves.items():
        ax.plot(df['precision'], df['recall'], 'o-', ms=3, label=variant)
    ax.set_xlabel('precision'); ax.set_ylabel('recall'); ax.legend(); plt.show()

    best_f05 = pd.DataFrame({v: df.loc[df['f05'].idxmax(), ['threshold', 'recall', 'precision', 'f05']] for v, df in curves.items()}).T
    print(f"Recall at precision >= {TARGET_PRECISION}:")
    print(pd.Series({v: df.loc[df['precision'] >= TARGET_PRECISION, 'recall'].max() for v, df in curves.items()}).to_string())
    print("\nBest F0.5 of each variant:")
    display(best_f05)

## Detection model: folding recall, length response and false positives

Turns a sliced theoretical spectrum into the histogram that OptimusPrimus is expected to report, with uncertainties and a goodness-of-fit. It needs the validation logs written by `evaluate_binned_efficiency` for the models in use (training cell above).

In [ ]:
MODE = 'seg_class'                              # 'seg' (segmentation only) or 'seg_class'
BIN_EDGES_NM = np.linspace(0., 16000., 17)      # one bin grid shared by theory, validation and measurement [nm]
BIN_MIDS_UM = 0.5 * (BIN_EDGES_NM[1:] + BIN_EDGES_NM[:-1]) / 1e3

### 1. Calibration on the annotated validation set

In [ ]:
logs = load_detection_logs(track_detector.detection_logs_path(MODE))
model = calibrate_detection_model(logs['gt_log'], logs['pred_log'], logs['pair_log'], BIN_EDGES_NM, logs['val_area_cm2'])

R, P, dR, dP = binned_recall_precision(model, return_errors=True)
calibration_table = pd.DataFrame({
    'bin_mid_um': BIN_MIDS_UM,
    'annotated': model['G'], 'recall': R.round(3), 'recall_err': dR.round(3),
    'detected': model['Q'], 'false_pos': model['FP'], 'precision': P.round(3), 'precision_err': dP.round(3)})

(b0, b1), (s0sq, s1sq) = model['bias'], model['var']
print(f"Validation area   : {model['area']:.4f} cm^2  ({len(model['pairs'])} matched pairs, "
      f"{model['n_pairs_rejected']} gross outliers excluded from the length response)")
print(f"Length bias       : {b0:+.0f} nm {b1:+.3f} * L")
print(f"Length resolution : sigma(L) = sqrt({np.sqrt(s0sq):.0f}^2 + ({np.sqrt(s1sq):.3f} L)^2) nm")
calibration_table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].errorbar(BIN_MIDS_UM, R, dR, fmt='o', label='recall (true length)')
ax[0].errorbar(BIN_MIDS_UM, P, dP, fmt='s', label='precision (reported length)')
ax[0].set_xlabel('length [um]'); ax[0].set_ylim(0, 1.05); ax[0].legend()

pairs_um = model['pairs'] / 1e3
L = np.linspace(0., BIN_EDGES_NM[-1], 200)
sig = np.sqrt(s0sq + s1sq * L ** 2)
ax[1].scatter(pairs_um[:, 0], pairs_um[:, 1] - pairs_um[:, 0], s=6, alpha=0.4)
ax[1].plot(L / 1e3, (b0 + b1 * L) / 1e3, 'r')
ax[1].fill_between(L / 1e3, (b0 + b1 * L - sig) / 1e3, (b0 + b1 * L + sig) / 1e3, color='r', alpha=0.2)
ax[1].set_xlabel('annotated length [um]'); ax[1].set_ylabel('reported - annotated [um]')
plt.tight_layout(); plt.show()

### 2. Sliced theoretical spectrum

**Demo only**: `counts_mg` is a toy shape, sliced on a reference area of 1 cm² and rescaled to the number of tracks found. With the real analysis, replace `counts_mg` with the PrimusCode spectrum (tracks per mg in each bin of `BIN_EDGES_NM`), slice it on `SAMPLE_AREA` and remove the rescaling line; with the toy spectrum the chi2 below only illustrates the mechanics.

In [ ]:
ellipses = track_detector.seg_cls_ellipses if MODE == 'seg_class' else track_detector.seg_ellipses
lengths_nm = ellipses['len_um'].to_numpy() * 1e3
SAMPLE_AREA = track_detector.tot_area_cm2        # area analysed by the pipeline [cm^2]
SAMPLE_DENSITY = 2.4                             # target density [g/cm^3]: set it for your sample (halite ~2.16)

x_mids = 0.5 * (BIN_EDGES_NM[1:] + BIN_EDGES_NM[:-1])
counts_mg = np.exp(-0.5 * ((x_mids - 6000.) / 1500.) ** 2)        # DEMO shape

SLICE_AREA = 1.0                                 # DEMO: shape only. Real analysis: SLICE_AREA = SAMPLE_AREA

# slice_spectrum is a Monte Carlo: average several realisations to suppress its own noise
N_unit = np.mean([slice_spectrum(counts_mg, sample_area_cm2=SLICE_AREA, sample_density_g_cm3=SAMPLE_DENSITY,
                                 x_bins=BIN_EDGES_NM, angular_pdf=None, phi_cut_deg=0.,
                                 pit_width=500., bulk_etching_depth=100., correction=True)
                  for _ in range(50)], axis=0)
N_i = N_unit * (len(lengths_nm) / N_unit.sum())                   # DEMO normalisation (remove with the real spectrum)
print(f"Sliced spectrum: {N_i.sum():.0f} tracks in {len(N_i)} bins")

### 3. Fold, compare with the measurement

In [ ]:
mu, C_cal = fold_detection(N_i, model, SAMPLE_AREA, fp_mode='density')
n_obs, _ = np.histogram(lengths_nm, bins=BIN_EDGES_NM)

groups = merge_bins_auto(mu, min_expected=10.)                    # Gaussian approximation needs >~10 counts per bin
chi2, ndof, pval = detection_chi2(*aggregate_bins(n_obs, mu, C_cal, groups))
print(f"chi2 / ndof = {chi2:.1f} / {ndof}   p = {pval:.3f}   "
      f"({len(groups)} merged bins; expected total {mu.sum():.0f}, observed {n_obs.sum()})")

In [ ]:
sigma_cal = np.sqrt(np.diag(C_cal))
edges_um = BIN_EDGES_NM / 1e3

fig, ax = plt.subplots(figsize=(7, 4))
ax.stairs(mu, edges_um, color='C0', label='model')
ax.fill_between(np.repeat(edges_um, 2)[1:-1], np.repeat(mu - sigma_cal, 2), np.repeat(mu + sigma_cal, 2),
                color='C0', alpha=0.25, label='calibration uncertainty')
ax.errorbar(BIN_MIDS_UM, n_obs, np.sqrt(np.maximum(n_obs, 1)), fmt='ko', ms=4, label='measured (Poisson errors)')
ax.set_xlabel('track length [um]'); ax.set_ylabel('tracks per bin'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# The precision-based variant assumes the same track density as the validation set
mu_p, _ = fold_detection(N_i, model, SAMPLE_AREA, fp_mode='precision')
print(f"expected reported tracks: false-positive-rate mode {mu.sum():.0f} | precision mode {mu_p.sum():.0f}")